# Week 21 Optional Deep-Dive: Three Airflow Concepts That Pay Off Later

This is an optional, async companion to the Week 21 main notebook.
You do not need to finish this to keep up with the course. It is here
for students who want to understand WHY some of the rules in the main
notebook exist, and to set you up for Week 22 (the drift -> retrain ->
redeploy loop).

We will cover three topics, each in roughly two cells:

1. XCom mechanics. What happens when a PythonOperator returns a value?
   Where does it go? Why is "pass S3 keys, not DataFrames" a rule and
   not a suggestion?
2. Datasets and data-aware scheduling. Airflow's way of saying "this
   DAG runs when this data is updated", instead of "every hour".
3. Sensors and deferrable mode. The difference between a sensor that
   holds a worker slot for an hour and one that suspends itself to a
   tiny async process while it waits.

How to use this notebook: read it linearly. Most code cells are
illustration-only and marked as such; running them is not required
(and several would need a local Airflow which we do not have).

Prerequisites: you understand DAGs, tasks, and PythonOperator at the
level of the main Week 21 notebook. Nothing else is required.

## 1. XCom mechanics: what actually happens when a task returns

In the main notebook we told you: pass S3 keys via XCom, never the
actual data. Here is the mechanical reason.

When a PythonOperator callable returns a value, Airflow:

1. Serializes the return value (by default, JSON; pickle is opt-in
   via the `enable_xcom_pickling` config setting, which MWAA leaves
   OFF).
2. Writes one row to the `xcom` table in the Airflow metadata database
   (Aurora PostgreSQL on MWAA).
3. Stores the row keyed by `(dag_id, run_id, task_id, key)`. The
   default key is `"return_value"`.

A downstream task fetches it with `ti.xcom_pull(task_ids="upstream")`,
which round-trips through the metadata DB again.

Three consequences that bite in practice:

- Default JSON serialization will reject most non-trivial Python
  objects. A pandas DataFrame is not JSON-serializable; you will get
  `TypeError: Object of type DataFrame is not JSON serializable`.
- The XCom row lives in the Aurora metadata DB. There is no hard
  Airflow-imposed cap by default (the underlying PostgreSQL `text`
  column can hold up to 1 GB), but every team that ships Airflow
  treats anything over roughly 1 MB as abuse: it slows down the
  scheduler loop, bloats backups, and makes the UI sluggish.
- XCom is not a data bus. It is metadata. The right pattern is: task
  A writes data to S3, returns the S3 key, task B pulls the key and
  reads S3 itself. The key is a few dozen bytes; the data stays in
  object storage where it belongs.

Custom XCom backends exist (subclasses of `BaseXCom` that store the
payload in S3 or GCS and put only a pointer in the metadata DB). MWAA
supports this via `core.xcom_backend` in the environment config. It
is a legitimate fix for teams that have inherited DAGs returning
medium-sized objects. For new DAGs the simpler rule is still: return
a reference, not the data.

Think About It: in Week 22 we are going to chain a "compute drift
metrics" task into a "decide whether to retrain" task. The drift
metrics output is small (a dict of floats). Is that a fine XCom
payload? Yes. Is the new training dataset that the retrain task
consumes a fine XCom payload? No - that goes to S3, and only the URI
goes through XCom.

In [ ]:
# What this shows: the difference between an XCom payload that is fine
# (a small dict of references) and one that is not (a DataFrame). This
# cell is illustration-only; it does not need a running Airflow.

# WRONG shape - returning the whole batch as a dict of lists. Will JSON-
# serialize on small data, but the row in the xcom table grows linearly
# with the batch size. At 100k rows this is several megabytes per task
# instance and the metadata DB starts to suffer.
def pull_batch_wrong(**context):
    import pandas as pd
    df = pd.read_csv("s3://bread-academy-shared/inbox/batch.csv")
    # Anti-pattern: shoving the data itself through XCom.
    return df.to_dict(orient="records")

# RIGHT shape - return only the S3 URI. The downstream task reads S3
# itself. XCom row is ~60 bytes regardless of how big the batch is.
def pull_batch_right(**context):
    s3_key = "inbox/batch.csv"
    bucket = "bread-academy-shared"
    # The reference is what flows through XCom.
    return {"bucket": bucket, "key": s3_key, "row_count_hint": 100_000}

# And how a downstream task consumes the right-shape XCom:
def invoke_endpoint(**context):
    ti = context["ti"]
    ref = ti.xcom_pull(task_ids="pull_batch")  # {"bucket": ..., "key": ...}
    # The downstream task reads S3 itself - the data never sat in
    # the Airflow metadata DB.
    import boto3
    s3 = boto3.client("s3")
    obj = s3.get_object(Bucket=ref["bucket"], Key=ref["key"])
    # ... process obj["Body"] ...

print("Pattern: return references via XCom, read the data from S3 in the consumer.")

## 2. Datasets: scheduling on data, not on the clock

Up to now every DAG you have seen is either `schedule=None` (trigger-
only, like all four Week 21 lab DAGs) or has a cron schedule
(`schedule="0 * * * *"`). Both are time-based mental models.

Airflow 2.4 added a third option: schedule a DAG on a Dataset. A
Dataset is a URI string that one task declares as its `outlet`. When
the task succeeds, Airflow marks the dataset as updated. Any DAG that
has that dataset in its `schedule=[...]` list is triggered.

The mental model: a Dataset is a typed channel between DAGs. Producer
DAGs put things in the channel (by succeeding a task whose `outlets`
include the dataset). Consumer DAGs subscribe to the channel.

Why this matters for Week 22:

- Today (Week 21) you trigger DAGs by hand from Databricks via
  `mwaa.invoke_rest_api`. That is fine for "I want to run a smoke
  test".
- In Week 22 we are going to want "when the drift detection DAG says
  drift is real, automatically run the retrain DAG". The wrong way
  to do that is to call `TriggerDagRunOperator` from the drift DAG;
  it couples the two DAGs and hides the dependency.
- The right way is to declare a Dataset like
  `Dataset("s3://bread-academy-shared/drift-alerts/")` that the drift
  DAG writes to as an outlet, and have the retrain DAG schedule on
  it. The dependency is visible in the Airflow UI's Datasets view
  and the two DAGs stay independent.

Airflow 2.10-specific additions you might see in docs:

- `DatasetAlias` (2.10+) lets you name a group of datasets and refer
  to it by alias in schedules. Useful when the actual dataset URI is
  computed at task time.
- `Metadata` (2.10+) lets a producer task attach extra info (e.g.
  row_count, partition_id) to the dataset event, which the consumer
  can read.

We will not use either of those in Week 22; the plain Dataset is
sufficient. Mention them so you recognize them in the Airflow UI.

In [ ]:
# What this shows: a minimal producer DAG and a minimal consumer DAG
# wired together by a single Dataset. Illustration-only - do not deploy
# these to MWAA; the lab DAGs the instructor already deployed cover the
# main notebook.

from datetime import datetime
from airflow import DAG
from airflow.datasets import Dataset
from airflow.operators.python import PythonOperator

# The "channel". Any task that lists this in its outlets will, on
# success, fire a dataset-updated event.
fraud_predictions_dataset = Dataset("s3://bread-academy-shared/fraud-predictions/")

def write_predictions(**context):
    # In real life: invoke the SageMaker endpoint, write results to S3
    # under the prefix that the dataset URI points to. The success of
    # this task is what makes Airflow consider the dataset updated.
    print("wrote predictions to s3://bread-academy-shared/fraud-predictions/run-X/")

# Producer DAG: runs hourly, declares the dataset as an outlet.
with DAG(
    dag_id="example_fraud_predictions_producer",
    start_date=datetime(2026, 1, 1),
    schedule="@hourly",
    catchup=False,
) as producer:
    PythonOperator(
        task_id="write_predictions",
        python_callable=write_predictions,
        outlets=[fraud_predictions_dataset],  # the dataset-updated trigger
    )

def evaluate_drift(**context):
    print("read latest predictions from S3, computed PSI, decided drift=False")

# Consumer DAG: NO cron. It runs when the dataset is updated.
with DAG(
    dag_id="example_drift_evaluator_consumer",
    start_date=datetime(2026, 1, 1),
    schedule=[fraud_predictions_dataset],
    catchup=False,
) as consumer:
    PythonOperator(
        task_id="evaluate_drift",
        python_callable=evaluate_drift,
    )

print("Producer DAG writes -> dataset is marked updated -> consumer DAG is triggered.")
print("In Week 22 the consumer will be the retrain DAG, gated by drift status.")

## 3. Sensors: poke vs reschedule vs deferrable

A sensor is a task that waits for a condition. Classic examples: wait
for a file to appear in S3, wait for an upstream system to finish.
Sensors are everywhere in production data platforms and they are also
the single biggest source of "why is my Airflow worker pool full
again" incidents.

Three modes:

- `mode="poke"` (default). The sensor occupies one worker slot the
  whole time it waits. Cheap to reason about, expensive at scale: a
  sensor that waits two hours pins one worker slot for two hours.
  If you have 10 sensors waiting and 8 worker slots, you have a
  scheduler deadlock waiting to happen.
- `mode="reschedule"`. Between pokes the task releases the worker
  slot and the scheduler re-queues it after `poke_interval` seconds.
  Better, but every wake-up still costs a fresh task instance startup
  (which on MWAA can be a few seconds). Good for slow-changing
  conditions where you check every few minutes.
- `deferrable=True` (Airflow 2.2+, supported on MWAA 2.7.2+). The
  sensor's "are we there yet" loop is handed off to a Triggerer
  process - a lightweight asyncio runner that can hold thousands of
  pending triggers simultaneously. No worker slot is occupied while
  waiting. When the trigger fires, the task resumes on a worker for
  whatever short post-condition work is needed.

The Amazon provider on MWAA 2.10 exposes `deferrable=True` on most
AWS sensors: `S3KeySensor`, `SqsSensor`, `EmrJobFlowSensor`,
`GlueJobSensor`, `BatchSensor`, and others. Setting it to True is a
one-character change with a very large operational payoff.

Rule of thumb:

- Waiting under 60 seconds: `mode="poke"` is fine.
- Waiting minutes to hours, condition can change at any moment:
  `deferrable=True` if the operator supports it, else
  `mode="reschedule"`.
- Never run a poke-mode sensor with `timeout` > 1 hour on MWAA. You
  are gambling with the worker pool.

MWAA-specific note: the Triggerer process runs as part of the MWAA
environment. You do not start it yourself. Deferrable sensors "just
work" on MWAA 2.7.2 and later, including the 2.10.x environment
this class uses.

In [ ]:
# What this shows: the same wait-for-an-S3-key requirement written
# three ways. The three sensors are functionally equivalent in what
# they wait for; they differ only in how they consume MWAA resources
# while waiting. Illustration-only - do not deploy.

from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor

with DAG(
    dag_id="example_sensor_modes",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
) as dag:

    # Mode 1 - poke. Holds a worker slot for the full timeout.
    # OK only for very short waits.
    wait_poke = S3KeySensor(
        task_id="wait_poke",
        bucket_name="bread-academy-shared",
        bucket_key="inbox/batch.csv",
        aws_conn_id="aws_default",
        poke_interval=10,
        timeout=60,
        mode="poke",
    )

    # Mode 2 - reschedule. Releases the worker slot between pokes.
    # Good for slow conditions checked every few minutes.
    wait_reschedule = S3KeySensor(
        task_id="wait_reschedule",
        bucket_name="bread-academy-shared",
        bucket_key="inbox/batch.csv",
        aws_conn_id="aws_default",
        poke_interval=300,
        timeout=timedelta(hours=2).total_seconds(),
        mode="reschedule",
    )

    # Mode 3 - deferrable. Hands the wait to the Triggerer process.
    # Zero worker-slot cost while waiting. This is the right default
    # for any meaningful wait on MWAA 2.10.
    wait_deferrable = S3KeySensor(
        task_id="wait_deferrable",
        bucket_name="bread-academy-shared",
        bucket_key="inbox/batch.csv",
        aws_conn_id="aws_default",
        deferrable=True,
        timeout=timedelta(hours=2).total_seconds(),
    )

print("All three wait for the same S3 key. Only the deferrable one")
print("scales to hundreds of concurrent waits without breaking MWAA.")

## Decision table: which Airflow primitive for which job

| Need | Right tool | Wrong tool |
|---|---|---|
| Pass a 1 KB dict between tasks | XCom (default) | S3 round-trip |
| Pass a 100 MB Parquet between tasks | S3 + XCom for the URI | XCom (will work briefly, then wreck the metadata DB) |
| Run DAG B after DAG A finishes, same schedule | `TriggerDagRunOperator` (rare) or just put both in one DAG | nothing |
| Run DAG B when DAG A produces new data | Dataset + `schedule=[dataset]` on DAG B | `TriggerDagRunOperator` from inside DAG A |
| Wait under a minute for a fast condition | sensor `mode="poke"` | deferrable (overkill) |
| Wait minutes to hours for an external event | sensor `deferrable=True` | sensor `mode="poke"` (will starve the worker pool) |
| Wait an unbounded amount of time | rethink the design | any sensor with no timeout |

The unifying principle: Airflow's metadata DB and worker pool are
both small, shared, contended resources. Anything you can keep OUT of
them (large data, long waits) belongs OUT of them.

## Where this points: Week 22

You will see all three of these primitives again in Week 22 when we
close the drift -> retrain -> redeploy loop:

- The drift detection DAG will compute PSI from the latest
  predictions, return the float result via XCom (small dict, correct
  use), and on "drift detected" write a marker file to an S3 prefix
  whose Dataset URI the retrain DAG subscribes to.
- The retrain DAG will not have a cron schedule. It will have
  `schedule=[drift_alert_dataset]`. It runs exactly when there is
  something to retrain on.
- A `SageMakerTrainingJobSensor` (or `SageMakerEndpointSensor`) with
  `deferrable=True` will wait for the training job and the endpoint
  rollout to complete, without holding worker slots for the 20-40
  minutes those take.

If you understood the three cells above, the Week 22 culmination DAG
will read like obvious composition rather than a new pile of
concepts.

## References

Airflow 2.10 docs:

- Data-aware scheduling: https://airflow.apache.org/docs/apache-airflow/2.10.5/authoring-and-scheduling/datasets.html
- Deferrable operators and triggers: https://airflow.apache.org/docs/apache-airflow/2.10.5/authoring-and-scheduling/deferring.html
- XCom (core concepts): https://airflow.apache.org/docs/apache-airflow/2.10.5/core-concepts/xcoms.html

Amazon provider on Airflow 2.10:

- S3KeySensor source and `deferrable` parameter: https://airflow.apache.org/docs/apache-airflow-providers-amazon/stable/_api/airflow/providers/amazon/aws/sensors/s3/index.html

AWS MWAA:

- MWAA 2.7.2+ deferrable operator support: https://aws.amazon.com/blogs/big-data/introducing-amazon-mwaa-support-for-apache-airflow-version-2-7-2-and-deferrable-operators/

Astronomer learn pages (well-written background):

- Pass data between tasks (XCom mechanics): https://www.astronomer.io/docs/learn/airflow-passing-data-between-tasks
- Custom XCom backend strategies: https://www.astronomer.io/docs/learn/custom-xcom-backend-strategies
- Datasets and data-aware scheduling: https://www.astronomer.io/docs/learn/airflow-datasets
- Airflow sensors guide: https://www.astronomer.io/docs/learn/what-is-a-sensor